In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if not (ROOT / "src" / "gop_empirical").is_dir():
    ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import torch

CKPT = ROOT / "checkpoints" / "e16" / "transformer_ckpt.pt"
CONFIG = ROOT / "configs" / "e_learned_scoring.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("ROOT", ROOT)
print("checkpoint", CKPT.is_file(), CKPT)
print("config", CONFIG.is_file(), CONFIG)
print("device", DEVICE)
assert CKPT.is_file(), f"missing {CKPT}; train E16 first (--features c8_lpp_lpr_embed)"
assert CONFIG.is_file(), CONFIG

ROOT D:\FSB_MSE\FINAL\GOP-Empirical-Study
checkpoint True D:\FSB_MSE\FINAL\GOP-Empirical-Study\checkpoints\e16\transformer_ckpt.pt
config True D:\FSB_MSE\FINAL\GOP-Empirical-Study\configs\e_learned_scoring.yaml
device cuda


In [2]:
from gop_empirical.scoring.checkpoint import load_checkpoint

ckpt = load_checkpoint(CKPT, device="cpu")
stored_test = (ckpt.get("metrics") or {}).get("test") or {}
print("experiment_id", ckpt["experiment_id"])
print("architecture ", ckpt["architecture"])
print("feature_set  ", ckpt["feature_set"])
print("model_kwargs ", ckpt["model_kwargs"])
print("fit          ", ckpt.get("fit"))
print("stored test PCC", stored_test.get("pcc"))

experiment_id E16
architecture  transformer
feature_set   c8_lpp_lpr_embed
model_kwargs  {'input_dim': 78, 'd_model': 32, 'nhead': 4, 'nlayers': 2, 'dim_feedforward': 64, 'dropout': 0.1, 'max_len': 50, 'n_phones': 39}
fit           {'best_epoch': 14, 'best_val_mse': 0.05205657075486804, 'epochs_ran': 22}
stored test PCC 0.6713800848773046


In [3]:
from gop_empirical.experiment import eval_group_e_checkpoint, load_config

cfg = load_config(CONFIG)
result = eval_group_e_checkpoint(
    cfg,
    experiment_id="E16",
    checkpoint_path=CKPT,
    package_root=ROOT,
    device_override=DEVICE,
    split="test",
)
m = result["metrics"]
print("checkpoint", result["checkpoint_path"])
print("feature_set", result["feature_set"], "arch", result["architecture"])
print("split", result["split"], "device", result["device"], "n", m["n"])
print(
    "E16 test  PCC", round(m["pcc"], 4),
    "SCC", round(m["scc"], 4),
    "MAE", round(m["mae"], 4),
    "MSE", round(m["mse"], 4),
)

E16 test  PCC=0.6714  SCC=0.4121  MAE=0.1325  MSE=0.0742  n=47369
checkpoint checkpoints/e16/transformer_ckpt.pt
feature_set c8_lpp_lpr_embed arch transformer
split test device cuda n 47369
E16 test  PCC 0.6714 SCC 0.4121 MAE 0.1325 MSE 0.0742


In [4]:
import pandas as pd

pd.DataFrame(
    [
        {
            "experiment": result["experiment_id"],
            "split": result["split"],
            "n": m["n"],
            "PCC": m["pcc"],
            "SCC": m["scc"],
            "MAE": m["mae"],
            "MSE": m["mse"],
        }
    ]
)

,experiment,split,n,PCC,SCC,MAE,MSE
0,E16,test,47369,0.67138,0.412075,0.132528,0.074204
